# Module 2.9 — Evaluation: How Good Is Your Model?

**Module 2: Data & ML Fundamentals | Week 2 | Lecture 6 of 6**

---

You've trained models. You've seen MSE and accuracy. But these are just the start. Today we learn how to **properly evaluate** a classifier — because "90% accuracy" isn't always as good as it sounds.

## Glossary

Today's vocabulary is all about measuring how good a model is. Lots of new acronyms — we'll use them by their full names where possible.

- **Accuracy** — the fraction of predictions that are correct. Simple but misleading when classes are imbalanced.
- **Confusion matrix** — a small table showing how many predictions ended up in each (predicted, actual) pair. Reveals what kinds of errors the model is making.
- **True positive (TP)** — model said yes, answer was yes.
- **True negative (TN)** — model said no, answer was no.
- **False positive (FP)** — model said yes, answer was no.
- **False negative (FN)** — model said no, answer was yes.
- **Precision** — of the times the model said yes, how often it was right. `TP / (TP + FP)`.
- **Recall** — of the actual yeses in the data, how many the model caught. `TP / (TP + FN)`.
- **F1-score** — a balance between precision and recall. Useful when you can't pick which one matters more.
- **Support** — how many examples of each class were in the test set. Helps you read precision/recall in context.
- **Learning curve** — a plot of model performance vs. training-set size. Tells you if more data would help.
- **Validation curve** — a plot of model performance vs. some hyperparameter (like model complexity). Tells you whether the model is over- or under-fitting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

---

## Accuracy: Simple But Misleading

**Accuracy** = fraction of predictions that are correct.

$$\text{Accuracy} = \frac{\text{correct predictions}}{\text{total predictions}}$$

Sounds great. But here's the problem.

In [ ]:
# Imagine a disease that affects 1% of people
# A lazy model that always predicts "healthy" gets...
n_people = 1000
n_sick = 10
n_healthy = 990

# Model that always predicts "healthy"
correct = n_healthy  # It gets all healthy people right
accuracy = correct / n_people

print(f"Always-predict-healthy accuracy: {accuracy:.1%}")
print(f"But it misses every single sick person!")

99% accuracy — and completely useless. This happens whenever classes are **imbalanced** (one class is much more common than another).

Accuracy is a fine starting point, but we need better tools.

---

## The Confusion Matrix

A confusion matrix shows exactly what your model got right and wrong. For binary classification, it's a 2×2 table:

|  | Predicted Negative | Predicted Positive |
|--|---|---|
| **Actually Negative** | True Negative (TN) | False Positive (FP) |
| **Actually Positive** | False Negative (FN) | True Positive (TP) |

- **True Positive (TP):** Model said yes, and it was right
- **True Negative (TN):** Model said no, and it was right
- **False Positive (FP):** Model said yes, but it was wrong ("false alarm")
- **False Negative (FN):** Model said no, but it was wrong ("missed it")

Let's see one in practice.

In [ ]:
# Create a binary classification problem
X, y = make_classification(
    n_samples=500, n_features=5, n_informative=3,
    n_redundant=0, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0, 0]}")
print(f"False Positives: {cm[0, 1]}")
print(f"False Negatives: {cm[1, 0]}")
print(f"True Positives:  {cm[1, 1]}")

In [ ]:
# Visualize it
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix')
plt.show()

Reading the confusion matrix:
- **Diagonal** (top-left, bottom-right) = correct predictions. You want these to be big.
- **Off-diagonal** = mistakes. You want these to be small.
- **Top-right (FP):** Model said class 1 but it was class 0
- **Bottom-left (FN):** Model said class 0 but it was class 1

---

## Multi-Class Confusion Matrix

For problems with more than two classes, the confusion matrix grows. Each row is the actual class, each column is the predicted class.

In [ ]:
# Iris: 3 classes
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42
)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=iris.target_names,
    ax=ax, cmap='Blues'
)
ax.set_title('Iris Classification — Confusion Matrix')
plt.show()

You can instantly see which classes the model confuses. If setosa is always correct but versicolor and virginica get mixed up, that tells you something about how similar those species are (at least in terms of these features).

---

## Classification Report

scikit-learn's `classification_report` gives you a summary of several metrics per class.

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

The key columns:
- **Precision:** Of all predictions for this class, how many were correct?
- **Recall:** Of all actual instances of this class, how many did we find?
- **F1-score:** A balance between precision and recall
- **Support:** How many test samples of this class

We're not going deep into precision/recall in this module (that's in the deep dive). The confusion matrix and accuracy are enough for now. Just know that these exist for when accuracy alone isn't sufficient.

---

## Visualizing Model Performance: Train vs Validation Curves

Last session we saw the overfitting curve — training error vs test error as model complexity changes. Let's look at another important plot: how performance changes **during training** (or with different amounts of data).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Vary model complexity (max_depth) and track train/test accuracy
depths = range(1, 21)
train_accs = []
test_accs = []

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=42
)

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_accs.append(tree.score(X_train, y_train))
    test_accs.append(tree.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_accs, 'bo-', label='Training accuracy')
plt.plot(depths, test_accs, 'ro-', label='Test accuracy')
plt.xlabel('Tree Depth (complexity)')
plt.ylabel('Accuracy')
plt.title('Validation Curve: Decision Tree on Iris')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Reading this plot:
- If train and test are both low → **underfitting** (model too simple)
- If train is high but test is lower → **overfitting** (model too complex)
- If both are high and close together → **good fit**

For Iris with a decision tree, a depth of 3-5 seems to be the sweet spot.

### Learning curves: how performance changes with data size

In [ ]:
# How does accuracy change as we add more training data?
train_sizes_pct = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
train_accs = []
test_accs = []

for pct in train_sizes_pct:
    # Create a split with a specific training fraction
    X_tr, X_te, y_tr, y_te = train_test_split(
        iris.data, iris.target, test_size=0.25, random_state=42
    )
    n = int(len(X_tr) * pct)
    if n < 3:  # Need at least a few samples
        n = 3
    X_sub = X_tr[:n]
    y_sub = y_tr[:n]

    tree = DecisionTreeClassifier(max_depth=4, random_state=42)
    tree.fit(X_sub, y_sub)
    train_accs.append(tree.score(X_sub, y_sub))
    test_accs.append(tree.score(X_te, y_te))

plt.figure(figsize=(8, 5))
actual_sizes = [max(3, int(len(X_tr) * p)) for p in train_sizes_pct]
plt.plot(actual_sizes, train_accs, 'bo-', label='Training accuracy')
plt.plot(actual_sizes, test_accs, 'ro-', label='Test accuracy')
plt.xlabel('Number of Training Samples')
plt.ylabel('Accuracy')
plt.title('Learning Curve: More Data Helps')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Notice:
- **Training accuracy starts at 100%** (easy to memorize a tiny dataset) and goes down slightly
- **Test accuracy starts low** and goes up as we add more training data
- **The gap narrows** with more data — more data reduces overfitting

---

## Putting It All Together: The Evaluation Checklist

When you train a classifier, always do these things:

1. **Check accuracy** — a quick sanity check
2. **Look at the confusion matrix** — what kinds of mistakes is it making?
3. **Compare train vs test performance** — is it overfitting?
4. **Think about whether accuracy is the right metric** — if classes are imbalanced, accuracy can be misleading

---

## Exercises

### Exercise 1: Read the confusion matrix

A spam classifier produces this confusion matrix:

```
                Predicted Not Spam   Predicted Spam
Actually Not Spam       180                 20
Actually Spam            30                 70
```

Answer these questions:
1. How many emails were there total?
2. What is the accuracy?
3. How many spam emails did the model miss (false negatives)?
4. How many non-spam emails were incorrectly flagged (false positives)?
5. Which type of error is worse for the user — false positives or false negatives?

In [ ]:
# Exercise 1: Your answers here


### Exercise 2: Build and evaluate

Load the Iris dataset. Train a `LogisticRegression` on it. Print the accuracy, display the confusion matrix, and print the classification report. Which class does the model struggle with most?

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Validation curve

Using the Iris dataset and `LogisticRegression`, vary the `C` parameter (try values: 0.001, 0.01, 0.1, 1, 10, 100). `C` controls regularization — smaller C = simpler model, larger C = more complex model.

Plot training accuracy vs test accuracy for each value of C. What value of C gives the best test accuracy?

In [ ]:
# Exercise 3: Your code here


---

## Summary

**Accuracy** measures overall correctness but can mislead with imbalanced classes.

**Confusion matrix** shows exactly what the model gets right and wrong. Diagonal = correct, off-diagonal = errors.

**Classification report** gives precision, recall, and F1 per class — useful when accuracy isn't enough.

**Validation curves** (train vs test as complexity changes) and **learning curves** (train vs test as data grows) are your best tools for diagnosing overfitting and underfitting.

**The evaluation checklist:** accuracy → confusion matrix → train/test comparison → is this the right metric?

---

**Next session (Lab):** Build a classifier on a real dataset (Iris or Titanic) and evaluate it thoroughly.